<a href="https://colab.research.google.com/github/attabeezy/crop-guard/blob/main/notebooks/03_train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CropGuard — class-weighted EfficientNetV2 training

This notebook trains the 22-class baseline using the immutable manifests produced by notebook #2. It performs frozen-backbone transfer learning followed by low-learning-rate fine-tuning.

**The held-out test set is not loaded or evaluated here.** Notebook #4 will use it once, after model and threshold decisions are complete.

## 1. Select a GPU runtime

In Colab choose **Runtime → Change runtime type → T4 GPU**, then run the next cell. Training on CPU is intentionally blocked.

In [ ]:
%pip install -q kagglehub pandas scikit-learn pillow

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 2026
IMAGE_SIZE = 224
BATCH_SIZE = 32
FROZEN_EPOCHS = 15
FINE_TUNE_EPOCHS = 10
UNFREEZE_LAST_LAYERS = 35

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No GPU detected. Select a T4 GPU runtime before training.")
print("TensorFlow:", tf.__version__)
print("GPU:", gpus[0])

## 2. Download and verify the immutable audit bundle

In [ ]:
import hashlib
import json
import shutil
import urllib.request
import zipfile

AUDIT_COMMIT = "4c8b366df4c251d6424258c3cb74a58f2289ee16"
AUDIT_URL = (
    "https://raw.githubusercontent.com/attabeezy/crop-guard/"
    f"{AUDIT_COMMIT}/cropguard-data-prep.zip"
)
EXPECTED_AUDIT_SHA256 = "1f69013b0a0fc58cba5dccab37d50e8767902580f443ac13daa3fba885e3269c"
audit_zip = Path("/content/cropguard-data-prep.zip")
audit_dir = Path("/content/cropguard-data-prep")

urllib.request.urlretrieve(AUDIT_URL, audit_zip)
actual_hash = hashlib.sha256(audit_zip.read_bytes()).hexdigest()
if actual_hash != EXPECTED_AUDIT_SHA256:
    raise RuntimeError(f"Audit ZIP hash mismatch: {actual_hash}")
if audit_dir.exists():
    shutil.rmtree(audit_dir)
with zipfile.ZipFile(audit_zip) as archive:
    archive.extractall(audit_dir)

recorded_hashes = json.loads((audit_dir / "split_hashes.json").read_text())
for filename, expected in recorded_hashes.items():
    actual = hashlib.sha256((audit_dir / filename).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"Manifest hash mismatch: {filename}")
audit = json.loads((audit_dir / "data_audit.json").read_text())
print("Audit bundle verified:", actual_hash)
display(audit)

## 3. Re-download the pinned CCMT mirror and resolve manifest paths

Only train and validation manifests are loaded. The test CSV remains sealed.

In [ ]:
import kagglehub

DATASET_HANDLE = "nirmalsankalana/crop-pest-and-disease-detection"
dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
train_frame = pd.read_csv(audit_dir / "train.csv")
validation_frame = pd.read_csv(audit_dir / "validation.csv")

def resolve_manifest(frame):
    result = frame.copy()
    result["absolute_path"] = result["path"].map(lambda value: str(dataset_root / value))
    missing = result[~result["absolute_path"].map(lambda value: Path(value).is_file())]
    if len(missing):
        raise RuntimeError(f"{len(missing)} manifest paths are missing; dataset version changed")
    return result

train_frame = resolve_manifest(train_frame)
validation_frame = resolve_manifest(validation_frame)
assert set(train_frame["path"]).isdisjoint(validation_frame["path"])
assert set(train_frame["duplicate_group"]).isdisjoint(validation_frame["duplicate_group"])
print(f"Train: {len(train_frame):,}; validation: {len(validation_frame):,}")

## 4. Verify image identity

The default verifies a deterministic sample because notebook #2 already hashed every image. Set `VERIFY_ALL_IMAGES = True` for a full 1.25 GB re-hash.

In [ ]:
VERIFY_ALL_IMAGES = False
VERIFY_SAMPLE_SIZE = 300

combined = pd.concat([train_frame, validation_frame], ignore_index=True)
to_verify = combined if VERIFY_ALL_IMAGES else combined.sample(
    n=min(VERIFY_SAMPLE_SIZE, len(combined)), random_state=SEED
)
mismatches = []
for row in to_verify.itertuples(index=False):
    actual = hashlib.sha256(Path(row.absolute_path).read_bytes()).hexdigest()
    if actual != row.sha256:
        mismatches.append(row.path)
if mismatches:
    raise RuntimeError(f"Image hash mismatches: {mismatches[:5]}")
print(f"Verified {len(to_verify):,} image hashes; mismatches: 0")

## 5. Establish class order and class weights

Class order is derived once, saved with the model, and later exported directly to Android. Balanced weights prevent large classes from dominating the loss.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = sorted(train_frame["label"].unique())
if len(classes) != 22 or set(classes) != set(validation_frame["label"].unique()):
    raise RuntimeError("Expected the same 22 classes in train and validation")
class_to_index = {label: index for index, label in enumerate(classes)}
train_frame["target"] = train_frame["label"].map(class_to_index)
validation_frame["target"] = validation_frame["label"].map(class_to_index)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(classes)),
    y=train_frame["target"].to_numpy(),
)
class_weights = {index: float(weight) for index, weight in enumerate(weights)}
weight_table = pd.DataFrame({
    "index": range(len(classes)),
    "label": classes,
    "train_count": [int((train_frame["target"] == i).sum()) for i in range(len(classes))],
    "class_weight": weights,
})
display(weight_table)

## 6. Build deterministic input pipelines

Augmentation is part of the model and is active only during training. Validation images are never augmented.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path, target):
    image = tf.io.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
    return image, target

def make_dataset(frame, training):
    dataset = tf.data.Dataset.from_tensor_slices((
        frame["absolute_path"].to_numpy(),
        frame["target"].to_numpy(dtype=np.int32),
    ))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    dataset = dataset.batch(BATCH_SIZE, drop_remainder=False)
    return dataset.prefetch(AUTOTUNE)

train_dataset = make_dataset(train_frame, training=True)
validation_dataset = make_dataset(validation_frame, training=False)
images, targets = next(iter(train_dataset))
print("Batch:", images.shape, targets.shape, images.dtype)

## 7. Build the model

EfficientNetV2B0 includes input rescaling internally and therefore receives pixels in the `[0, 255]` range produced above.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.08, fill_mode="reflect", seed=SEED + 1),
    tf.keras.layers.RandomZoom(0.12, fill_mode="reflect", seed=SEED + 2),
    tf.keras.layers.RandomTranslation(0.08, 0.08, fill_mode="reflect", seed=SEED + 3),
    tf.keras.layers.RandomContrast(0.15, seed=SEED + 4),
], name="training_augmentation")

backbone = tf.keras.applications.EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    include_preprocessing=True,
)
backbone.trainable = False

inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3), name="image")
x = augmentation(inputs)
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(x)
x = tf.keras.layers.Dropout(0.30, seed=SEED, name="classifier_dropout")(x)
outputs = tf.keras.layers.Dense(
    len(classes), activation="softmax", name="probabilities"
)(x)
model = tf.keras.Model(inputs, outputs, name="cropguard_efficientnetv2b0")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_accuracy"),
    ],
)
model.summary()

## 8. Phase A — train the classifier head

In [ ]:
artifact_dir = Path("/content/cropguard_training_artifacts")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
artifact_dir.mkdir()

def callbacks(checkpoint_name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            artifact_dir / checkpoint_name,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=1
        ),
        tf.keras.callbacks.TerminateOnNaN(),
    ]

frozen_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=FROZEN_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks("best_frozen.keras"),
)

## 9. Phase B — fine-tune the upper backbone

Batch-normalization layers remain frozen to protect their ImageNet statistics with the relatively small minority classes.

In [ ]:
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE_LAST_LAYERS]:
    layer.trainable = False
for layer in backbone.layers[-UNFREEZE_LAST_LAYERS:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_accuracy"),
    ],
)
print("Trainable parameters after unfreezing:",
      sum(np.prod(v.shape) for v in model.trainable_weights))

fine_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks("best_finetuned.keras"),
)

## 10. Save training metadata and inspect validation history

These are tuning results, not final test metrics.

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone

def history_frame(history, phase):
    frame = pd.DataFrame(history.history)
    frame.insert(0, "epoch", np.arange(1, len(frame) + 1))
    frame.insert(0, "phase", phase)
    return frame

history = pd.concat([
    history_frame(frozen_history, "frozen"),
    history_frame(fine_history, "fine_tune"),
], ignore_index=True)
history.to_csv(artifact_dir / "training_history.csv", index=False)
weight_table.to_csv(artifact_dir / "class_weights.csv", index=False)

display_names = {
    "healthy": "Healthy",
    "fall_armyworm": "Fall armyworm damage",
    "grasshopper": "Grasshopper damage",
    "leaf_beetle": "Leaf beetle damage",
    "green_mite": "Green mite damage",
    "leaf_miner": "Leaf miner damage",
}
with (artifact_dir / "labels.txt").open("w", encoding="utf-8") as stream:
    for label in classes:
        crop, key = label.split("|", 1)
        condition = display_names.get(key, key.replace("_", " ").title())
        stream.write(f"{crop}|{key}|{crop.title()} — {condition}\n")

training_config = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "tensorflow_version": tf.__version__,
    "audit_commit": AUDIT_COMMIT,
    "audit_sha256": EXPECTED_AUDIT_SHA256,
    "dataset_handle": DATASET_HANDLE,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "class_count": len(classes),
    "class_order": classes,
    "train_images": len(train_frame),
    "validation_images": len(validation_frame),
    "test_images_used": 0,
    "unfrozen_backbone_layers": UNFREEZE_LAST_LAYERS,
}
(artifact_dir / "training_config.json").write_text(json.dumps(training_config, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history["loss"], label="train")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[1].plot(history["accuracy"], label="train")
axes[1].plot(history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(artifact_dir / "training_curves.png", dpi=160)
plt.show()
display(history.tail())

## 11. Package and download the training artifacts

Keep this ZIP unchanged. It contains both best checkpoints, exact label order, weights, configuration, history, and curves. It does not contain the CCMT images.

In [ ]:
archive = shutil.make_archive(
    "/content/cropguard-training-artifacts", "zip", artifact_dir
)
archive_hash = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
print(f"Created {archive}")
print(f"Size: {Path(archive).stat().st_size / 1_000_000:.1f} MB")
print(f"SHA-256: {archive_hash}")

In [ ]:
from google.colab import files
files.download("/content/cropguard-training-artifacts.zip")

## Next step

Provide `cropguard-training-artifacts.zip` for inspection. Notebook #4 will compare the frozen and fine-tuned checkpoints on validation, select one without touching the test set, then perform the single held-out evaluation and confidence calibration.